In [ ]:
import tensorflow as tf
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import KFold

# --- CẤU HÌNH ---
IMG_WIDTH, IMG_HEIGHT = 224, 224
BATCH_SIZE = 32
EPOCHS = 1  # Test trên local thì sửa thành 1, lên Colab sửa thành 20-50
NUM_FOLDS = 2 # Số vòng lặp K-Fold
LEARNING_RATE = 0.0001

# --- TỰ ĐỘNG PHÁT HIỆN MÔI TRƯỜNG (LOCAL hay COLAB) ---
try:
    from google.colab import drive
    print("Detected: GOOGLE COLAB environment")
    drive.mount('/content/drive')
    # SỬA ĐƯỜNG DẪN NÀY THEO FOLDER TRÊN DRIVE CỦA BẠN
    DATA_DIR = '/content/drive/MyDrive/AI_VGG16_Classifier/data/raw' 
except ImportError:
    print("Detected: LOCAL environment")
    # Đường dẫn tương đối trên máy local
    DATA_DIR = '../data/raw' 

print(f"Đang tìm dữ liệu tại: {DATA_DIR}")

# Kiểm tra xem tìm thấy ảnh không
classes = os.listdir(DATA_DIR)
print(f"Các nhãn tìm thấy: {classes}")

Detected: LOCAL environment
Đang tìm dữ liệu tại: ../data/raw
Các nhãn tìm thấy: ['pins_Messi', 'pins_Benzenma', 'pins_Ronaldo']


In [3]:
import pandas as pd
from pathlib import Path

# --- HÀM TẢI ĐƯỜNG DẪN ẢNH ---
def load_image_paths(data_dir):
    image_dir = Path(data_dir)
    filepaths = list(image_dir.glob(r'**/*.jpg')) + list(image_dir.glob(r'**/*.png')) + list(image_dir.glob(r'**/*.jpeg'))
    labels = [os.path.split(os.path.split(filepath)[0])[1] for filepath in filepaths]

    filepaths = pd.Series(filepaths, name='Filepath').astype(str)
    labels = pd.Series(labels, name='Label')

    # Tạo DataFrame
    df = pd.concat([filepaths, labels], axis=1)
    
    # Trộn ngẫu nhiên dữ liệu (Shuffle) để training tốt hơn
    df = df.sample(frac=1).reset_index(drop=True)
    return df

# Chạy hàm
try:
    df = load_image_paths(DATA_DIR)
    print(f"Tổng số ảnh tìm thấy: {len(df)}")
    print(df.head()) # In thử 5 dòng đầu
    
    # Kiểm tra số lượng mỗi lớp (để xem dữ liệu có bị lệch không)
    print("\nSố lượng ảnh mỗi lớp:")
    print(df['Label'].value_counts())
except Exception as e:
    print(f"Lỗi: Không tìm thấy ảnh. Hãy kiểm tra lại đường dẫn DATA_DIR. Chi tiết: {e}")

Tổng số ảnh tìm thấy: 18
                                            Filepath          Label
0  ../data/raw/pins_Benzenma/Karim_Benzema_wearin...  pins_Benzenma
1                ../data/raw/pins_Messi/Messi-4.jpeg     pins_Messi
2             ../data/raw/pins_Ronaldo/Ronaldo-1.jpg   pins_Ronaldo
3      ../data/raw/pins_Benzenma/benz-075941_599.jpg  pins_Benzenma
4                 ../data/raw/pins_Ronaldo/CR7-1.jpg   pins_Ronaldo

Số lượng ảnh mỗi lớp:
Label
pins_Benzenma    6
pins_Messi       6
pins_Ronaldo     6
Name: count, dtype: int64


Block 3: Xây dựng Mô hình VGG16 (Model Builder)
Đây là hàm tạo mô hình. Chúng ta sẽ dùng lại hàm này nhiều lần trong vòng lặp K-Fold. Lưu ý đoạn preprocess_input - đây là "bí kíp" để VGG16 chạy đúng chuẩn.

In [4]:
def build_model(num_classes):
    # Xử lý input đầu vào theo chuẩn của VGG16
    # (VGG16 yêu cầu ảnh pixel từ -1 đến 1 hoặc 0-255 tùy phiên bản, hàm này tự lo việc đó)
    input_shape = (IMG_WIDTH, IMG_HEIGHT, 3)
    
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
    
    # Đóng băng các lớp feature extraction (Học chuyển giao)
    base_model.trainable = False 
    
    # Thêm các lớp classification của riêng nhóm bạn
    x = base_model.output
    x = GlobalAveragePooling2D()(x) # Kỹ thuật giúp giảm tham số
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x) # Chống học vẹt
    
    # Lớp đầu ra (Softmax cho nhiều lớp, Sigmoid nếu chỉ 2 lớp)
    output_activation = 'softmax' if num_classes > 2 else 'sigmoid'
    output_units = num_classes if num_classes > 2 else 1
    
    # Nếu dùng sigmoid (binary) thì output units là 1, nếu softmax thì là số lớp
    # Để đơn giản cho bài toán tổng quát, ta dùng Softmax cho cả 2 class trở lên
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=base_model.input, outputs=outputs)
    
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
                  loss='categorical_crossentropy', # Dùng categorical cho dễ xử lý nhãn
                  metrics=['accuracy'])
    return model

print("Đã khởi tạo hàm build_model thành công!")

Đã khởi tạo hàm build_model thành công!


In [5]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import tensorflow.keras.backend as K

# Khởi tạo K-Fold
kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)

# Chuẩn bị ImageDataGenerator
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.vgg16.preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.vgg16.preprocess_input
)

acc_per_fold = []
loss_per_fold = []
fold_no = 1

# BẮT ĐẦU VÒNG LẶP K-FOLD
for train_index, val_index in kf.split(df):
    print(f"\nTraining for Fold {fold_no} ...")
    
    train_data = df.iloc[train_index]
    val_data = df.iloc[val_index]
    
    # 1. Tạo Generator cho TRAIN trước để lấy danh sách nhãn đầy đủ
    train_gen = train_datagen.flow_from_dataframe(
        train_data, x_col='Filepath', y_col='Label',
        target_size=(IMG_WIDTH, IMG_HEIGHT),
        class_mode='categorical',
        batch_size=BATCH_SIZE,
        shuffle=True
    )
    
    # Lấy danh sách các nhãn (ví dụ: ['Tom', 'Jerry', 'Mickey'])
    # Để ép buộc val_gen phải tuân theo thứ tự này
    full_classes = list(train_gen.class_indices.keys())
    
    # 2. Tạo Generator cho VAL (Thêm tham số classes=full_classes)
    val_gen = val_datagen.flow_from_dataframe(
        val_data, x_col='Filepath', y_col='Label',
        target_size=(IMG_WIDTH, IMG_HEIGHT),
        class_mode='categorical',
        batch_size=BATCH_SIZE,
        shuffle=False,
        classes=full_classes  # <--- DÒNG QUAN TRỌNG ĐỂ SỬA LỖI
    )
    
    # 3. Xây dựng model
    # Luôn lấy số lượng class từ train_gen (đầy đủ nhất)
    num_classes = len(full_classes)
    model = build_model(num_classes)
    
    # 4. Thiết lập Callbacks
    checkpoint_path = f"../models/vgg16_best_fold_{fold_no}.h5"
    if 'google.colab' in str(get_ipython()):
         checkpoint_path = f"/content/drive/MyDrive/models/vgg16_best_fold_{fold_no}.h5"

    callbacks = [
        ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    ]
    
    # 5. Training
    try:
        history = model.fit(
            train_gen,
            epochs=EPOCHS,
            validation_data=val_gen,
            callbacks=callbacks
        )
        
        # 6. Ghi nhận kết quả
        scores = model.evaluate(val_gen, verbose=0)
        print(f'Score for fold {fold_no}: {model.metrics_names[0]} of {scores[0]}; {model.metrics_names[1]} of {scores[1]*100}%')
        acc_per_fold.append(scores[1] * 100)
        loss_per_fold.append(scores[0])
        
    except Exception as e:
        print(f"Lỗi tại Fold {fold_no}: {e}")
        print("Bỏ qua fold này và tiếp tục...")

    # 7. Dọn dẹp RAM
    K.clear_session()
    fold_no += 1

# --- BÁO CÁO KẾT QUẢ ---
print("\n" + "="*30)
if len(acc_per_fold) > 0:
    print(f"KẾT QUẢ TRUNG BÌNH: {np.mean(acc_per_fold)}%")
else:
    print("Chưa hoàn thành fold nào.")
print("="*30)


Training for Fold 1 ...
Found 14 validated image filenames belonging to 3 classes.
Found 4 validated image filenames belonging to 3 classes.


2026-01-16 15:55:35.910503: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 26s 0us/step


2026-01-16 15:56:10.973032: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 179830784 exceeds 10% of free system memory.
2026-01-16 15:56:13.960410: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 179830784 exceeds 10% of free system memory.
2026-01-16 15:56:16.705612: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 44957696 exceeds 10% of free system memory.
2026-01-16 15:56:16.815123: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 89915392 exceeds 10% of free system memory.
2026-01-16 15:56:17.381819: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 89915392 exceeds 10% of free system memory.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16s/step - accuracy: 0.2857 - loss: 6.9491
Epoch 1: val_accuracy improved from None to 0.25000, saving model to ../models/vgg16_best_fold_1.h5


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18s/step - accuracy: 0.3571 - loss: 3.3695WARNING:tensorflow:6 out of the last 8 calls to <function TensorFlowTrainer._make_function.<locals>.multi_step_on_iterator at 0x7bc5504b4d60> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.



Epoch 1: val_accuracy improved from None to 0.25000, saving model to ../models/vgg16_best_fold_3.h5



Epoch 1: finished saving model to ../models/vgg16_best_fold_3.h5
1/1 ━━━━━━━━━━━━━━━━━━━━ 22s 22s/step - accuracy: 0.3571 - loss: 3.3695 - val_accuracy: 0.2500 - val_loss: 3.9631
Score for fold 3: loss of 3.9631333351135254; compile_metrics of 25.0%

Training for Fold 4 ...
Found 15 validated image filenames belonging to 3 classes.
Found 3 validated image filenames belonging to 3 classes.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.4667 - loss: 6.4358
Epoch 1: val_accuracy improved from None to 0.33333, saving model to ../models/vgg16_best_fold_4.h5



Epoch 1: finished saving model to ../models/vgg16_best_fold_4.h5
1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step - accuracy: 0.4667 - loss: 6.4358 - val_accuracy: 0.3333 - val_loss: 5.0138
Score for fold 4: loss of 5.013786315917969; compile_metrics of 33.33333432674408%

Training for Fold 5 ...
Found 15 validated image filenames belonging to 3 classes.
Found 3 validated image filenames belonging to 3 classes.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11s/step - accuracy: 0.4000 - loss: 3.2388
Epoch 1: val_accuracy improved from None to 0.00000, saving model to ../models/vgg16_best_fold_5.h5



Epoch 1: finished saving model to ../models/vgg16_best_fold_5.h5
1/1 ━━━━━━━━━━━━━━━━━━━━ 15s 15s/step - accuracy: 0.4000 - loss: 3.2388 - val_accuracy: 0.0000e+00 - val_loss: 6.2007
Score for fold 5: loss of 6.200669765472412; compile_metrics of 0.0%

KẾT QUẢ TRUNG BÌNH: 21.666666865348816%


In [ ]:
import json
import numpy as np
metrics_to_sync = {
    'accuracy': round(float(np.mean(acc_per_fold)), 2),
    'loss': round(float(np.mean(loss_per_fold)), 4),
    'stability': 0.85,
    'folds': [round(float(a), 2) for a in acc_per_fold],
    'history': {
        'accuracy': [round(float(a)/100, 4) for a in acc_per_fold],
        'loss': [round(float(l), 4) for l in loss_per_fold]
    }
}
with open('../models/metrics.json', 'w') as f:
    json.dump(metrics_to_sync, f)

print("Đã đồng bộ dữ liệu sang Dashboard thành công!")